# Chapter 9: tank days and contribution per tank day

Reads a synthetic tank occupancy file (one row per fill as a date interval), checks that no tank holds two fills at once, counts the tanks full on each day by overlapping intervals and prints contribution per tank day by beer. The table shape follows the occupancy file of the author's "Brew What Sells with AI" workshop dataset; the figures are Kestrel's.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

occ = pd.read_csv("../data/tank_occupancy.csv", parse_dates=["occupied_from", "occupied_to"])
beers = pd.read_csv("../data/beers.csv").set_index("beer")
occ.head()

## Overlapping intervals

Never join on a single date. A fill occupies its tank for weeks; ask which intervals cover the day.

In [ ]:
clash = []
for tank, g in occ.sort_values("occupied_from").groupby("tank"):
    prev_end = g.occupied_to.shift()
    clash += list(g.loc[g.occupied_from < prev_end, "batch_code"])
print("Fills that overlap another on the same tank:", clash or "none")

days = pd.date_range("2026-01-01", "2026-12-31")
full = [((occ.occupied_from <= d) & (occ.occupied_to > d)).sum() for d in days]
print(f"Tanks full on an average day: {sum(full) / len(days):.1f} of 15 ({100 * sum(full) / len(days) / 15:.1f} percent in 2026)")
print(f"Busiest day: {max(full)} tanks")

## Contribution per tank day (chapter 9.2)

In [ ]:
HL_A_FILL = 574.2  # 600 hL cast / 1.045
beers["contribution_usd_hl"] = beers.net_price_usd_hl - beers.variable_cost_usd_hl
occ["contribution_usd"] = HL_A_FILL * occ.beer.map(beers.contribution_usd_hl)
per_day = occ.groupby("beer")[["contribution_usd", "days"]].sum()
per_day["usd_per_tank_day"] = per_day.contribution_usd / per_day.days
print(per_day.usd_per_tank_day.round(0))
print(f"Pale ale against lager: {per_day.usd_per_tank_day['Pale ale'] / per_day.usd_per_tank_day['Lager']:.1f} times")
print(f"Lager tank day: USD {per_day.usd_per_tank_day['Lager']:,.0f}, INR {round(per_day.usd_per_tank_day['Lager']) * 90:,.0f}, EUR {round(per_day.usd_per_tank_day['Lager']) / 1.1:,.0f}")

## The year in tank days (chapter 9.1) and planned against actual (chapter 9.3)

In [ ]:
year = beers.copy()
year["dispatched_hl"] = 12 * year.dispatched_hl_month
year["cast_hl"] = 1.045 * year.dispatched_hl
year["fills"] = year.cast_hl / 600
year["tank_days"] = year.fills * (year.days_in_tank + year.turnaround_days)
supply = 15 * 365
print(year[["dispatched_hl", "cast_hl", "fills", "tank_days"]].round(1))
used = year.tank_days.sum()
print(f"Used {used:,.0f} of {supply:,} tank days: {100 * used / supply:.1f} percent")
blended = 100000 * 80 / round(used)
print(f"Blended contribution per tank day: USD {blended:,.0f}")

plan = occ.groupby("beer")[["planned_days", "days"]].mean()
plan["overrun_a_fill"] = plan.days - plan.planned_days
plan["tank_days_lost"] = plan.overrun_a_fill * year.fills
print(plan.round(1))
lost = round(plan.tank_days_lost.sum())
print(f"Lost {lost} tank days; the plan would use {round(used) - lost:,} ({100 * (round(used) - lost) / supply:.1f} percent); lager share {100 * plan.tank_days_lost['Lager'] / plan.tank_days_lost.sum():.0f} percent")
print("Overrun reasons in the file:")
print(occ.loc[occ.days > occ.planned_days, "overrun_reason"].value_counts())

## Cash in the tanks (chapter 9.4) and the fermenter you did not buy (chapter 9.5)

In [ ]:
hl_days = year.fills * 600 * year.days_in_tank
avg_cash = hl_days.sum() * 27 / 365
print(f"hL-days {hl_days.sum():,.0f}; average cash in tanks USD {avg_cash:,.0f}; cost of money at 12 percent USD {0.12 * avg_cash:,.0f}")
print(f"Lager share of the cash: {100 * hl_days['Lager'] / hl_days.sum():.0f} percent")
print(f"The {lost} overrun days hold USD {lost * 600 * 27 / 365 * 0.12:,.0f} a year in cost of money")
freed = round(round(year.fills.sum(), 1) * 0.2, 1)
print(f"One day freed on each of {year.fills.sum():.1f} fills, one in five sold: {freed} days x {blended:,.0f} = USD {freed * round(blended):,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colours = {"Lager": "#2f7f72", "Pale ale": "#cfe6e1", "Stout": "#0b4f45"}
q1 = occ[occ.occupied_from < "2026-04-01"]
for _, r in q1.iterrows():
    ax.barh(r.tank, (r.occupied_to - r.occupied_from).days, left=r.occupied_from, color=colours[r.beer], edgecolor="white")
ax.set_xlim(pd.Timestamp("2026-01-01"), pd.Timestamp("2026-04-01"))
ax.invert_yaxis()
ax.set_title("Tank occupancy, first quarter (synthetic). Gaps are idle tank days.")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colours.values()]
ax.legend(handles, colours.keys(), loc="lower right")
plt.tight_layout()
plt.show()